# Project pipeline — step 0: environment + repos

This notebook sets up everything you need to reproduce the **baseline R2Gen** report generator, then add:

1) **RadGraph-based factuality metric + RL fine-tuning**  
2) **MC Dropout uncertainty estimation + calibration**  
3) *(Optional)* **CheXpert→NIH domain-shift AUROC**

> **Expected folder layout**
```
project_root/
  notebooks/   (these .ipynb files)
  repos/
    R2Gen/
  data/
    iu_xray/
    chexpert/
    nih_cxr14/
  outputs/
```


## 0.1 GPU check (recommended)

In [1]:
import torch, os, platform
print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Python: 3.13.5
Torch: 2.8.0+cpu
CUDA available: False


## 0.2 Install dependencies

This is the **simplest** pip-based setup that works with **RadGraph (torch>=2.1)** and usually works with R2Gen.

If you have a constrained environment, create a fresh conda env first (optional).

In [2]:
# If running in Colab, you may want:
# !pip -q install --upgrade pip

# Core
!pip -q install numpy pandas tqdm scikit-learn matplotlib seaborn pillow opencv-python

# Torch + vision (if not already)
# NOTE: If you need a specific CUDA build, install it following pytorch.org instructions.
!pip -q install torch torchvision --upgrade

# NLP + eval
!pip -q install nltk rouge-score sacrebleu bert-score

# RadGraph package (for both extraction + F1-RadGraph metric)
# (Stanford-AIMI/radgraph provides RadGraph and F1RadGraph classes)
!pip -q install git+https://github.com/Stanford-AIMI/radgraph.git



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
  DEPRECATION: Building 'rouge-score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge-score'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
  DEPRECATION: Building 'radgraph' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future vers

## 0.3 Clone R2Gen

In [3]:
import os, pathlib, textwrap, subprocess, sys

ROOT = pathlib.Path('.').resolve()
REPOS = ROOT/'repos'
REPOS.mkdir(exist_ok=True)

R2GEN_DIR = REPOS/'R2Gen'
if not R2GEN_DIR.exists():
    !git clone https://github.com/cuhksz-nlp/R2Gen.git {str(R2GEN_DIR)}
else:
    print('R2Gen already exists:', R2GEN_DIR)

print('Repo contents:', list(R2GEN_DIR.iterdir())[:8])


Repo contents: [WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/.git'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/.gitignore'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/data'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/main.py'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/models'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/modules'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/pycocoevalcap'), WindowsPath('C:/Users/sinab/Downloads/repos/R2Gen/README.md')]


Cloning into 'C:\Users\sinab\Downloads\repos\R2Gen'...


## 0.4 One-time NLTK data

In [4]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sinab\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

## 0.5 Set global paths used by the rest of notebooks

In [5]:
from pathlib import Path
ROOT = Path('.').resolve()
DATA = ROOT/'data'
OUT = ROOT/'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

paths = {
    "ROOT": str(ROOT),
    "R2GEN_DIR": str((ROOT/'repos'/'R2Gen').resolve()),
    "IU_XRAY_DIR": str((DATA/'iu_xray').resolve()),
    "CHEXPERT_DIR": str((DATA/'chexpert').resolve()),
    "NIH_CXR14_DIR": str((DATA/'nih_cxr14').resolve()),
    "OUT_DIR": str(OUT.resolve()),
}
paths


{'ROOT': 'C:\\Users\\sinab\\Downloads',
 'R2GEN_DIR': 'C:\\Users\\sinab\\Downloads\\repos\\R2Gen',
 'IU_XRAY_DIR': 'C:\\Users\\sinab\\Downloads\\data\\iu_xray',
 'CHEXPERT_DIR': 'C:\\Users\\sinab\\Downloads\\data\\chexpert',
 'NIH_CXR14_DIR': 'C:\\Users\\sinab\\Downloads\\data\\nih_cxr14',
 'OUT_DIR': 'C:\\Users\\sinab\\Downloads\\outputs'}

✅ Next: open **01_IU_XRay_Data.ipynb** to download/prepare the IU X-Ray dataset and validate file structure.